In [2]:
import os
import pandas as pd
import ast
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, f1_score, roc_auc_score, recall_score

In [3]:
# Veri seti yüklenir. "One_Hot_Encoded" kolonu çıkarılmıştır.
df = pd.read_csv(r"file_path")

# 'sequence' ve 'One_Hot_Encoded' kolonlarını çıkar
df = df.drop(["sequence", "numeric_sequence"], axis=1)

# AAC ve DPC sütunları işlenir.
for col in ["AAC", "DPC"]:
    col_df = df[col].apply(lambda x: pd.Series(ast.literal_eval(x)))
    col_df = col_df.rename(columns=lambda x: f"{col}_{str(x)}")
    df = pd.concat([df.drop(col, axis=1), col_df], axis=1)

# Sütun isimleri string formatına dönüştürülür ve sayısal veri türüne çevirilir.
df.columns = [str(col) for col in df.columns]
for col in df.columns:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Eksik değerleri doldur (NaN değerler için)
df.fillna(0, inplace=True)

# Özellikleri ve hedef değişkeni ayır
X = df.drop("anticancer", axis=1)
y = df["anticancer"]

# Veri setini eğitim ve test setlerine ayır
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Özellik ölçeklendirme
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [4]:
param_grid = {
    'n_estimators': [40, 50, 60],  # Ağaç sayısı
    'max_features': ['sqrt', "log2"],  # Maksimum özellik sayısı
    'max_depth': [12, 18, 24],  # Maksimum derinlik
    'min_samples_split': [5, 10, 15],  # Minimum örnek bölme sayısı
    'min_samples_leaf': [1, 2, 3],  # Yaprak düğümde minimum örnek sayısı
    'criterion': ["gini", "entropy"],  # Kalite ölçüm fonksiyonu
    'bootstrap': [True],  # Bootstrap kullanımı
    'oob_score': [True, False],  # OOB skorunu değerlendirme
    'warm_start': [True, False]  # Isıtılmış başlangıç
}

In [5]:
# Random Forest modeli oluşturma
rf = RandomForestClassifier(random_state=42)

# Stratified K-Fold çapraz doğrulama yöntemi
stratified_kfold = StratifiedKFold(n_splits=5)

# Grid Search ile hiperparametre optimizasyonu
grid_search = GridSearchCV(estimator=rf, param_grid=param_grid, cv=stratified_kfold, n_jobs=-1, verbose=2)
grid_search.fit(X_train, y_train)

# En iyi parametreleri ve skoru yazdırma
print("En İyi Parametreler:", grid_search.best_params_)
print("En İyi Skor:", grid_search.best_score_)

Fitting 5 folds for each of 1296 candidates, totalling 6480 fits
En İyi Parametreler: {'bootstrap': True, 'criterion': 'gini', 'max_depth': 18, 'max_features': 'sqrt', 'min_samples_leaf': 2, 'min_samples_split': 10, 'n_estimators': 50, 'oob_score': True, 'warm_start': True}
En İyi Skor: 0.882822376738306


In [6]:
param_grid = {
    'n_estimators': [49, 50, 51],  # Ağaç sayısı
    'max_features': ['sqrt', "log2"],  # Maksimum özellik sayısı
    'max_depth': [17, 18, 19],  # Maksimum derinlik
    'min_samples_split': [9, 10, 11],  # Minimum örnek bölme sayısı
    'min_samples_leaf': [1, 2, 3],  # Yaprak düğümde minimum örnek sayısı
    'criterion': ["gini"],  # Kalite ölçüm fonksiyonu
    'bootstrap': [True],  # Bootstrap kullanımı
    'oob_score': [True],  # OOB skorunu değerlendirme
    'warm_start': [True]  # Isıtılmış başlangıç
}

In [7]:
# Random Forest modeli oluşturma
rf = RandomForestClassifier(random_state=42)

# Stratified K-Fold çapraz doğrulama yöntemi
stratified_kfold = StratifiedKFold(n_splits=5)

# Grid Search ile hiperparametre optimizasyonu
grid_search = GridSearchCV(estimator=rf, param_grid=param_grid, cv=stratified_kfold, n_jobs=-1, verbose=2)
grid_search.fit(X_train, y_train)

# En iyi parametreleri ve skoru yazdırma
print("En İyi Parametreler:", grid_search.best_params_)
print("En İyi Skor:", grid_search.best_score_)

Fitting 5 folds for each of 162 candidates, totalling 810 fits
En İyi Parametreler: {'bootstrap': True, 'criterion': 'gini', 'max_depth': 18, 'max_features': 'sqrt', 'min_samples_leaf': 2, 'min_samples_split': 10, 'n_estimators': 50, 'oob_score': True, 'warm_start': True}
En İyi Skor: 0.882822376738306
